# Day 8 目标：实现 Retriever Routing

今天你要完成：

根据 task_type 控制检索范围，让 paper 问题只查论文资料、experiment 问题只查实验资料、dataset 问题只查数据集资料。

也就是从普通 RAG：

用户问题 → 查全部向量库 → 返回相似文档

升级为 Agentic RAG：

用户问题 → classify_task 得到 task_type → 根据 task_type 选择检索范围 → 返回对应类型文档

# 一、今天最终效果

假设用户问：

OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

如果 task_type = dataset_recommendation，Retriever 应该只查：

source_type = dataset_doc

而不是查到论文笔记或实验说明。

再比如用户问：

coco_val_n300_g1 这个实验的目的是什么？

如果 task_type = experiment_analysis，Retriever 应该只查：

source_type = experiment_doc

# 二、Day 8 要新增的文件

今天新增：

F:\ResearchAgent
├── src
│   └── research_agent
│       └── rag
│           └── retriever.py
│
└── scripts
    └── test_retriever_routing.py

今天暂时不改 LangGraph 节点，Day 9 再接入 nodes.py。

# 三、先写 retriever.py

路径：

F:\ResearchAgent\src\research_agent\rag\retriever.py

复制下面代码：

from typing import Dict, List, Optional

from langchain_core.documents import Document

from .indexer import load_vector_store
from .schemas import (
    SOURCE_TYPE_PAPER,
    SOURCE_TYPE_EXPERIMENT,
    SOURCE_TYPE_DATASET,
)


TASK_TYPE_TO_SOURCE_TYPE = {
    "paper_question": SOURCE_TYPE_PAPER,
    "experiment_analysis": SOURCE_TYPE_EXPERIMENT,
    "dataset_recommendation": SOURCE_TYPE_DATASET,

    # 兼容一些简写
    "paper": SOURCE_TYPE_PAPER,
    "experiment": SOURCE_TYPE_EXPERIMENT,
    "dataset": SOURCE_TYPE_DATASET,
}


def get_source_type_for_task(task_type: str) -> Optional[str]:
    """
    根据 task_type 决定应该检索哪类资料。

    返回：
    - paper_question -> paper_note
    - experiment_analysis -> experiment_doc
    - dataset_recommendation -> dataset_doc
    - code_help / general / report_generation -> None
    """
    return TASK_TYPE_TO_SOURCE_TYPE.get(task_type)


def build_metadata_filter(task_type: str) -> Optional[Dict]:
    """
    构建 Chroma metadata filter。

    Chroma 的简单过滤形式：
    {"source_type": "dataset_doc"}

    如果返回 None，表示不使用 metadata filter，检索全部文档。
    """
    source_type = get_source_type_for_task(task_type)

    if source_type is None:
        return None

    return {
        "source_type": source_type
    }


def retrieve_documents(
    query: str,
    task_type: str,
    top_k: int = 3,
    use_filter: bool = True,
) -> List[Document]:
    """
    根据 query 和 task_type 检索相关文档。

    参数：
    - query: 用户问题
    - task_type: 阶段一 classify_task 得到的任务类型
    - top_k: 返回前几个结果
    - use_filter: 是否启用 metadata filter

    返回：
    - List[Document]
    """
    vector_store = load_vector_store()

    metadata_filter = build_metadata_filter(task_type) if use_filter else None

    if metadata_filter:
        docs = vector_store.similarity_search(
            query=query,
            k=top_k,
            filter=metadata_filter,
        )
    else:
        docs = vector_store.similarity_search(
            query=query,
            k=top_k,
        )

    return docs


def document_to_dict(doc: Document) -> Dict:
    """
    把 LangChain Document 转成普通 dict，方便后续放进 AgentState。
    """
    return {
        "content": doc.page_content,
        "metadata": dict(doc.metadata),
    }


def retrieve_documents_as_dicts(
    query: str,
    task_type: str,
    top_k: int = 3,
    use_filter: bool = True,
) -> List[Dict]:
    """
    检索文档，并转成 dict 列表。

    Day 9 接入 LangGraph 时，AgentState 里建议保存 dict，
    而不是直接保存 Document 对象。
    """
    docs = retrieve_documents(
        query=query,
        task_type=task_type,
        top_k=top_k,
        use_filter=use_filter,
    )

    return [document_to_dict(doc) for doc in docs]


def format_retrieved_docs(
    docs: List[Document],
    max_chars_per_doc: int = 500,
) -> str:
    """
    把检索到的 Document 格式化成可读文本。
    后续可以给 final_answer 或 LLM prompt 使用。
    """
    if not docs:
        return "未检索到相关资料。"

    formatted_parts = []

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata
        content = doc.page_content[:max_chars_per_doc].replace("\n", " ")

        source = metadata.get("path", "unknown")
        source_type = metadata.get("source_type", "unknown")
        title = metadata.get("title", "")
        dataset = metadata.get("dataset", "")
        run_tag = metadata.get("run_tag", "")

        header_parts = [
            f"[{i}]",
            f"source_type={source_type}",
            f"path={source}",
        ]

        if title:
            header_parts.append(f"title={title}")

        if dataset:
            header_parts.append(f"dataset={dataset}")

        if run_tag:
            header_parts.append(f"run_tag={run_tag}")

        header = " | ".join(header_parts)

        formatted_parts.append(
            f"{header}\n{content}"
        )

    return "\n\n".join(formatted_parts)


def extract_sources_from_docs(docs: List[Document]) -> List[Dict]:
    """
    从检索结果中提取 sources，去重后用于最终回答展示。
    """
    seen = set()
    sources = []

    for doc in docs:
        metadata = doc.metadata
        path = metadata.get("path", "unknown")

        if path in seen:
            continue

        seen.add(path)

        sources.append({
            "path": path,
            "source_type": metadata.get("source_type", "unknown"),
            "title": metadata.get("title", ""),
            "dataset": metadata.get("dataset", ""),
            "run_tag": metadata.get("run_tag", ""),
        })

    return sources

# 四、代码重点解释
1. TASK_TYPE_TO_SOURCE_TYPE
TASK_TYPE_TO_SOURCE_TYPE = {
    "paper_question": SOURCE_TYPE_PAPER,
    "experiment_analysis": SOURCE_TYPE_EXPERIMENT,
    "dataset_recommendation": SOURCE_TYPE_DATASET,
}

它是 Day 8 的核心映射表。

意思是：

task_type	source_type
paper_question	paper_note
experiment_analysis	experiment_doc
dataset_recommendation	dataset_doc

后面 LangGraph 分类出来的是 task_type，但 Chroma metadata 里存的是 source_type。

所以中间必须有这个转换。

2. build_metadata_filter()
def build_metadata_filter(task_type: str) -> Optional[Dict]:
    source_type = get_source_type_for_task(task_type)

    if source_type is None:
        return None

    return {
        "source_type": source_type
    }

比如：

build_metadata_filter("dataset_recommendation")

返回：

{"source_type": "dataset_doc"}

后面传给 Chroma：

vector_store.similarity_search(
    query=query,
    k=top_k,
    filter={"source_type": "dataset_doc"},
)

这样 Chroma 只会在 dataset_doc 类型的 chunk 里查。

3. 为什么 code_help / general 返回 None？

因为你现在 Day 5 没准备 code 文档。

所以：

code_help
report_generation
general

暂时可以不做强过滤。

默认逻辑是：

source_type 无法映射 → 不加 filter → 查全部

后续你可以改成：

code_help → code_doc
report_generation → paper_note + experiment_doc
general → 全部

但 Day 8 先别复杂化。

# 五、创建测试脚本

路径：

F:\ResearchAgent\scripts\test_retriever_routing.py

复制下面代码：

from pathlib import Path
import sys


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))


from research_agent.rag.retriever import (
    build_metadata_filter,
    retrieve_documents,
    format_retrieved_docs,
    extract_sources_from_docs,
)


TEST_CASES = [
    {
        "query": "Guardrail-Agnostic 这篇论文关注什么问题？",
        "task_type": "paper_question",
        "expected_source_type": "paper_note",
    },
    {
        "query": "coco_val_n300_g1 这个实验的目的是什么？",
        "task_type": "experiment_analysis",
        "expected_source_type": "experiment_doc",
    },
    {
        "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
        "task_type": "dataset_recommendation",
        "expected_source_type": "dataset_doc",
    },
    {
        "query": "GQA 对场景图和关系分析有什么价值？",
        "task_type": "dataset_recommendation",
        "expected_source_type": "dataset_doc",
    },
]


def main():
    print("=" * 80)
    print("Test Retriever Routing")
    print("=" * 80)

    for case in TEST_CASES:
        query = case["query"]
        task_type = case["task_type"]
        expected_source_type = case["expected_source_type"]

        metadata_filter = build_metadata_filter(task_type)

        print("\n" + "=" * 80)
        print(f"Query: {query}")
        print(f"Task Type: {task_type}")
        print(f"Metadata Filter: {metadata_filter}")
        print(f"Expected Source Type: {expected_source_type}")
        print("=" * 80)

        docs = retrieve_documents(
            query=query,
            task_type=task_type,
            top_k=3,
            use_filter=True,
        )

        if not docs:
            print("No documents retrieved.")
            continue

        all_match = all(
            doc.metadata.get("source_type") == expected_source_type
            for doc in docs
        )

        print(f"Retrieved {len(docs)} documents.")
        print(f"All source_type match expected: {all_match}")

        print("\nRetrieved Docs:")
        print(format_retrieved_docs(docs, max_chars_per_doc=300))

        print("\nSources:")
        sources = extract_sources_from_docs(docs)
        for source in sources:
            print(f"- {source}")


if __name__ == "__main__":
    main()
    

# 六、运行测试前先确认索引存在

Day 8 使用 Day 7 的 Chroma 索引，所以先确认：

F:\ResearchAgent\storage\chroma_db

存在。

如果不确定，先运行：

cd F:\ResearchAgent
.\.conda\python.exe scripts\build_index.py

然后运行 Day 8 测试：

.\.conda\python.exe scripts\test_retriever_routing.py

# 七、预期输出

你希望看到类似：

Query: OpenImages-MIAP 的性别标注是图像级还是 bbox 级？
Task Type: dataset_recommendation
Metadata Filter: {'source_type': 'dataset_doc'}
Expected Source Type: dataset_doc

Retrieved 3 documents.
All source_type match expected: True

对实验问题：

Query: coco_val_n300_g1 这个实验的目的是什么？
Task Type: experiment_analysis
Metadata Filter: {'source_type': 'experiment_doc'}
Expected Source Type: experiment_doc

Retrieved 3 documents.
All source_type match expected: True

对论文问题：

Query: Guardrail-Agnostic 这篇论文关注什么问题？
Task Type: paper_question
Metadata Filter: {'source_type': 'paper_note'}
Expected Source Type: paper_note

Retrieved 3 documents.
All source_type match expected: True

只要 All source_type match expected: True，Day 8 的核心逻辑就跑通了。

# 八、你今天要理解的关键点
1. Day 7 的检索是“全库检索”

Day 7 你做的是：

similarity_search(query, k=3)

它会在所有 chunk 里找相似内容：

paper_note
experiment_doc
dataset_doc

都可能返回。

2. Day 8 的检索是“按任务类型过滤检索”

Day 8 改成：

similarity_search(
    query=query,
    k=3,
    filter={"source_type": "dataset_doc"},
)

这会先限制范围：

只在 dataset_doc 里查

然后再做语义相似度排序。

所以它是：

metadata filter + vector similarity

不是单纯关键词搜索。

3. 为什么这是 Agentic RAG？

普通 RAG 是：

用户问题 → 检索全部资料 → 生成回答

你的阶段二设计是：

用户问题
↓
classify_task 得到 task_type
↓
task_type 决定 source_type
↓
Retriever 只查对应资料库
↓
生成回答

也就是说：

Agent 的 Planner / Router 决定 RAG 查哪里。

这就比普通 RAG 更像 Agent 工作流。

# 九、常见问题
问题 1：为什么返回的文档 source_type 正确，但内容不一定最相关？

因为今天只做 metadata filter。

比如 dataset_recommendation 会保证只查 dataset_doc，但在 dataset_doc 内部，embedding 排序可能仍然不完美。

这和 embedding 模型能力有关。

你现在用的 all-MiniLM-L6-v2 对中文不是最强，先跑通就行。

问题 2：为什么 top_k=3 可能返回同一个文件的多个 chunk？

这是正常的。

因为 Chroma 存的是 chunk，不是整篇文档。

一个文件被切成多个 chunk 后，用户问题可能同时匹配到同一个文件的多个片段。

所以我们写了：

extract_sources_from_docs()

用于 Sources 去重。

问题 3：为什么要把 Document 转成 dict？

Day 9 接入 LangGraph 时，AgentState 更适合保存普通结构：

retrieved_docs: list[dict]

而不是直接保存 Document 对象。

因为 dict 更容易：

打印
序列化
调试
保存日志
传给 final_answer
后续做 API 输出

所以提前准备了：

retrieve_documents_as_dicts()

# 十、可选优化：加入 score

如果你想看相似度分数，可以额外在 retriever.py 加一个函数：

def retrieve_documents_with_scores(
    query: str,
    task_type: str,
    top_k: int = 3,
    use_filter: bool = True,
):
    vector_store = load_vector_store()

    metadata_filter = build_metadata_filter(task_type) if use_filter else None

    if metadata_filter:
        results = vector_store.similarity_search_with_score(
            query=query,
            k=top_k,
            filter=metadata_filter,
        )
    else:
        results = vector_store.similarity_search_with_score(
            query=query,
            k=top_k,
        )

    return results

它返回：

[(Document(...), score), (Document(...), score)]

但这个不是今天必须项。

# 十一、Day 8 验收标准

今天完成后，你应该做到：

1. 新增 src/research_agent/rag/retriever.py
2. 新增 scripts/test_retriever_routing.py
3. 能根据 task_type 生成 metadata filter
4. paper_question 只返回 paper_note
5. experiment_analysis 只返回 experiment_doc
6. dataset_recommendation 只返回 dataset_doc
7. 能格式化 retrieved docs
8. 能提取去重 Sources

核心验收命令：

cd F:\ResearchAgent
.\.conda\python.exe scripts\test_retriever_routing.py

核心验收结果：

All source_type match expected: True